# Assignment 4 – ML Lifecycle & AI Systems

## Question 1: The Complete ML Lifecycle vs Traditional SDLC

The Machine Learning (ML) lifecycle is the end-to-end process of building, deploying, and maintaining an ML system. Unlike traditional software, ML systems are data-driven and probabilistic, requiring a unique set of iterative steps.

### Stages of the ML Lifecycle

1. **Problem Definition:** Translate the business objective into a clear, measurable ML problem. Define what success looks like (e.g., reduce churn by 10%).
2. **Data Collection:** Gather relevant data from databases, APIs, sensors, or web scraping. More and higher-quality data generally leads to better models.
3. **Data Preprocessing & EDA:** Handle missing values, remove duplicates, explore distributions, visualize correlations, and understand the data before modeling.
4. **Feature Engineering:** Transform raw data into meaningful features that help the model learn patterns. This step heavily influences model performance.
5. **Model Selection & Training:** Choose an appropriate algorithm (e.g., Random Forest, Neural Network), train it on the prepared data, and tune hyperparameters.
6. **Model Evaluation:** Assess performance using metrics like accuracy, F1-score, and AUC-ROC on unseen validation/test data.
7. **Deployment:** Deploy the model as an API or embedded service so it can serve real-time or batch predictions in production.
8. **Monitoring & Retraining:** Continuously monitor for data drift, performance degradation, and retrain the model as needed to maintain accuracy.

### How ML Lifecycle Differs from Traditional SDLC

| Dimension | Traditional SDLC | ML Lifecycle |
|---|---|---|
| **Determinism** | Deterministic outputs (code behaves exactly as written) | Probabilistic outputs depending on data quality |
| **Iteration** | Relatively linear (Waterfall or Agile sprints) | Highly iterative — cycling back repeatedly |
| **Data Dependency** | Logic-driven; developer defines the rules | Data-driven; model learns rules from data |
| **Success Metrics** | Feature completion and bug counts | Statistical metrics (accuracy, recall, AUC) |
| **Post-deployment** | Stays correct unless code changes | Can degrade silently over time (drift) |

## Question 2: Why Problem Framing is the Most Critical Step

Translating a business need into a correctly framed data science problem is the most critical step in the ML lifecycle because every subsequent decision — data collection, algorithm choice, and evaluation metric — depends on how the problem is defined.

### Why It Matters

- If the problem type is wrong (e.g., treating a ranking problem as binary classification), no amount of data or compute will produce useful results.
- Choosing the wrong target variable means the model optimizes for something misaligned with business goals.
- The wrong metric (e.g., accuracy instead of recall in medical diagnosis) can create a model that passes evaluation but fails in production.

### Example: Poor Problem Framing Leading to Wasted Resources

A retail company wants to *"reduce customer churn."* A data science team incorrectly frames this as: *"Predict which customers will churn next month"* and builds a binary classifier. They achieve 92% accuracy and deploy. However, the business team wanted to **prevent churn** by proactively offering discounts — not just predict it. Because the model had no propensity score or churn probability, the marketing team couldn't prioritize which customers to target, and the intervention system was never built.

**Result:** 3 months and significant budget were spent on a model that generated no business value, purely because the framing focused on prediction accuracy rather than actionable intervention.

## Question 3: Predictive ML Models vs Generative AI Models

### Predictive ML Models

Predictive ML models are trained to map input features to a specific output label or value. They learn patterns from historical data to make predictions about future or unseen data. The output is constrained — a class label, a number, or a probability. They are **discriminative** in nature.

| Use Case | Description |
|---|---|
| **Credit Scoring** | Banks classify loan applicants as low/high risk based on income, credit history, and employment |
| **Disease Diagnosis** | Hospitals predict whether a patient has a condition like diabetes or cancer |
| **Demand Forecasting** | E-commerce platforms predict product demand using historical sales, seasonality, and promotions |

### Generative AI Models

Generative AI models learn the underlying distribution of training data and can generate entirely **new, original content** — text, images, audio, or code. They are not limited to a predefined output space. Examples include LLMs (GPT, Claude), image generators (DALL-E, Stable Diffusion), and audio synthesis models.

| Use Case | Description |
|---|---|
| **Automated Content Writing** | LLMs generate marketing copy, product descriptions, and blog articles from a short prompt |
| **Code Generation** | Tools like GitHub Copilot suggest and auto-complete code, accelerating developer productivity |
| **Drug Discovery** | Generative chemistry models create novel molecular structures, speeding up pharmaceutical research |

## Question 4: Structured, Unstructured & Multimodal Data

- **Structured Data:** Data organized in a predefined format with clear rows and columns, typically stored in relational databases or spreadsheets. Each field has a specific data type and meaning.
  - *Examples:* Customer transaction records (`customer_id`, `amount`, `date`), hospital patient records, CSV files with product inventory.

- **Unstructured Data:** Data that lacks a predefined format or schema. It cannot be easily stored in traditional rows and columns and requires specialized processing (NLP, computer vision).
  - *Examples:* Emails, social media posts, medical X-ray/MRI images, audio recordings, video surveillance footage.

- **Multimodal Data:** Data that combines two or more different data types (modalities) — text + image, or audio + video + text. Modern AI systems increasingly work with multimodal data for richer understanding.
  - *Examples:* Medical report combining doctor's notes (text) + scan images, a YouTube video with frames + audio + subtitles, an e-commerce listing with title (text) + product photo (image) + user review (text).

## Question 5: Cleaning & Preparing a Dataset with Issues

### Step 1: Initial Exploration

Begin with Exploratory Data Analysis (EDA). Use `df.info()`, `df.describe()`, and `df.isnull().sum()` to understand the shape of the data, identify missing value locations, check data types, and spot obvious anomalies.

In [ ]:
import pandas as pd
import numpy as np

# Load data
df = pd.read_csv('dataset.csv')

# Initial exploration
print(df.info())
print(df.describe())
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nDuplicate rows: {df.duplicated().sum()}")

### Step 2: Handle Missing Values (~20%)

The strategy depends on the percentage and type of missing data:
- **Numerical columns:** Impute with mean (if normally distributed) or median (if skewed).
- **Categorical columns:** Impute with mode or a new category `'Unknown'`.
- **High missingness (>40-50%):** Consider dropping the column entirely.
- **MNAR (Missing Not At Random):** Use `KNNImputer` or `IterativeImputer` for more accuracy.

In [ ]:
from sklearn.impute import KNNImputer

# Numerical: fill with median (robust to skew)
df['age'] = df['age'].fillna(df['age'].median())

# Categorical: fill with mode
df['category'] = df['category'].fillna(df['category'].mode()[0])

# Drop columns with >50% missing
threshold = 0.5
df = df[df.columns[df.isnull().mean() < threshold]]

# KNN imputation for remaining numerical columns
imputer = KNNImputer(n_neighbors=5)
numerical_cols = df.select_dtypes(include=[np.number]).columns
df[numerical_cols] = imputer.fit_transform(df[numerical_cols])

### Step 3: Remove Duplicate Rows

Duplicates bias training by making the model see certain patterns more often than they actually occur.

In [ ]:
# Remove duplicates
before = len(df)
df = df.drop_duplicates()
after = len(df)
print(f"Removed {before - after} duplicate rows.")

### Step 4: Handle Imbalanced Class Distribution

Imbalanced classes (e.g., 95% not-fraud vs 5% fraud) cause the model to predict the majority class for all samples. Strategies:
- **Oversampling:** Use SMOTE to generate synthetic minority samples.
- **Undersampling:** Randomly reduce majority class samples.
- **Class weights:** Pass `class_weight='balanced'` to algorithms like Logistic Regression or Random Forest.
- **Metrics:** Evaluate with F1-score, Precision-Recall AUC, or ROC-AUC instead of raw accuracy.

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split

X = df.drop('target', axis=1)
y = df['target']

# Split BEFORE resampling to prevent leakage
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Apply SMOTE only on training data
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("Class distribution after SMOTE:")
print(pd.Series(y_train_resampled).value_counts())

### Step 5: Final Preparation

- Encode categorical variables (Label Encoding or One-Hot Encoding).
- Scale numerical features using `StandardScaler` or `MinMaxScaler`.
- Split into train/validation/test sets **AFTER** all cleaning to prevent data leakage.

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

# One-hot encode categorical columns
df = pd.get_dummies(df, columns=['category'], drop_first=True)

# Scale numerical features
scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

print("Data preparation complete. Shape:", df.shape)

## Question 6: Tokenization, Embeddings & Context Window

- **Tokenization:** The process of breaking raw text into smaller units called tokens before feeding it to an NLP or LLM model. Tokens can be words, subwords, or characters. For example, `'I love machine learning'` might be tokenized into `['I', 'love', 'machine', 'learn', '##ing']` using a subword tokenizer like BPE (Byte Pair Encoding). This allows models to handle vocabulary efficiently, including unseen words by breaking them into known subword pieces.

- **Embeddings:** Dense numerical vector representations of tokens in a high-dimensional space. Embeddings capture semantic meaning — words with similar meanings have vectors that are geometrically close (e.g., `'king'` and `'queen'` are near each other). Examples: Word2Vec, BERT embeddings, OpenAI `text-embedding-ada-002`.

- **Context Window:** The maximum number of tokens an LLM can process in a single input/output interaction. It defines how much text the model can "see" at once. If a document exceeds the context window, it must be chunked. For example, GPT-4 supports up to 128,000 tokens.

### Why Embeddings are Critical for GenAI Systems

| Application | Role of Embeddings |
|---|---|
| **Semantic Search (RAG)** | Find relevant documents via similarity even when exact keywords don't match |
| **Memory & Context** | Store/retrieve conversation history in vector databases (Pinecone, Chroma, Weaviate) |
| **Multimodal Alignment** | Project text and images into a shared vector space (e.g., CLIP) |
| **Efficiency** | Compress complex information into compact vectors for fast large-scale similarity comparisons |

In [ ]:
# Example: Tokenization and embeddings using HuggingFace transformers
from transformers import AutoTokenizer
import numpy as np

# Load a tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

text = "I love machine learning and FastAPI"
tokens = tokenizer.tokenize(text)
token_ids = tokenizer.encode(text)

print(f"Text: {text}")
print(f"Tokens: {tokens}")
print(f"Token IDs: {token_ids}")
print(f"Number of tokens: {len(tokens)}")

## Question 7: Train/Validation/Test Split, Cross-Validation & Data Leakage

- **Train/Validation/Test Split:** The dataset is divided into three non-overlapping subsets:
  - **Training set (60–70%):** Used to fit the model.
  - **Validation set (10–20%):** Used for hyperparameter tuning and model selection during development.
  - **Test set (10–20%):** Held out entirely; used only once at the end for an unbiased estimate of real-world performance.

- **Cross-Validation:** A technique to make better use of limited data. In k-fold cross-validation, the dataset is split into *k* equal folds. The model trains on *k-1* folds and evaluates on the remaining fold, repeating *k* times. The final score is the average across all evaluations — giving a more reliable, lower-variance estimate.

- **Data Leakage:** Occurs when information from outside the training set (e.g., from the test set or future data) inadvertently influences the model during training. This causes an overly optimistic but false picture of model performance.

### Why Data Leakage is Dangerous

A model with data leakage passes evaluation with high scores, gets deployed, and then **immediately underperforms** because the information it relied on is unavailable in real-world scenarios:

- **Financial models** appear profitable in backtesting but lose money in live trading.
- **Medical diagnostic models** seem highly accurate but misdiagnose patients in clinical settings.
- **Fraud models** "predict" fraud perfectly because they were trained on post-fraud labels.

**Common causes:** Scaling before splitting (test set influences the scaler), including future information in features, or target encoding computed across the full dataset.

In [ ]:
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import numpy as np

# --- CORRECT split: split BEFORE any preprocessing ---
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Fit scaler ONLY on training data — then transform val/test
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit here
X_val_scaled   = scaler.transform(X_val)         # only transform
X_test_scaled  = scaler.transform(X_test)         # only transform

# --- k-Fold Cross-Validation ---
model = RandomForestClassifier(random_state=42)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=kf, scoring='f1')

print(f"CV F1 scores per fold: {cv_scores.round(3)}")
print(f"Mean F1: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

## Question 8: Model Evaluation Metrics

| Metric | Formula | Best Used When |
|---|---|---|
| **Accuracy** | (TP+TN)/(TP+TN+FP+FN) | Classes are balanced |
| **Precision** | TP/(TP+FP) | Cost of false positive is high (e.g., spam filter) |
| **Recall** | TP/(TP+FN) | Cost of false negative is high (e.g., cancer diagnosis) |
| **F1-Score** | 2×(P×R)/(P+R) | Imbalanced datasets; both FP and FN matter |
| **ROC-AUC** | Area under TPR vs FPR curve | Comparing models across all thresholds |

### Why Accuracy is Misleading in Fraud Detection

In fraud detection, the dataset is extremely imbalanced — typically **99% legitimate** and **1% fraud**. A naive model that predicts *"not fraud"* for every transaction achieves 99% accuracy without learning anything useful. It misses **100% of actual fraud cases** (Recall = 0), which is catastrophic. The business cares about catching fraud, not the accuracy number. In such scenarios, **Recall, Precision-Recall AUC, and F1-score** are far more meaningful.

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report,
                             confusion_matrix)
import matplotlib.pyplot as plt
import seaborn as sns

# Train and predict (example)
model = RandomForestClassifier(class_weight='balanced', random_state=42)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

# Compute metrics
print("=== Model Evaluation Metrics ===")
print(f"Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision : {precision_score(y_test, y_pred):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred):.4f}")
print(f"F1-Score  : {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC   : {roc_auc_score(y_test, y_prob):.4f}")
print()
print(classification_report(y_test, y_pred))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## Question 9: Model Drift — Data Drift, Concept Drift & Retraining

### 1. Data Drift (Covariate Shift)
Occurs when the **statistical properties of input features** change over time compared to what the model was trained on. The relationship between features and target may remain the same, but the distribution of incoming data has shifted.

*Example:* A fraud detection model trained on pre-COVID spending patterns encounters completely different behaviors post-COVID (more online shopping, less travel).

### 2. Concept Drift
Occurs when the **underlying relationship between input features and the target variable** changes over time. Even if data distribution stays similar, what constitutes fraud, churn, or a good recommendation has fundamentally changed.

*Example:* Customer preferences shift as new competitors enter the market. A recommendation model trained on old preferences now recommends products customers no longer want.

### 3. Monitoring Techniques

| Technique | Description |
|---|---|
| **Statistical Tests** | PSI and Kolmogorov-Smirnov tests to detect feature distribution changes |
| **Performance Tracking** | Log and alert on drops in precision, recall, or AUC vs baseline |
| **Data Quality Monitoring** | Track null rates, value ranges, and schema changes in pipelines |
| **Shadow Deployment** | Run new model in parallel to compare outputs without serving predictions |
| **Tools** | MLflow, Evidently AI, WhyLogs, AWS SageMaker Model Monitor |

### 4. Retraining Strategy

- **Scheduled Retraining:** Retrain on a fixed schedule (weekly, monthly) regardless of drift signals.
- **Triggered Retraining:** Monitor metrics and trigger retraining when a key metric drops below a threshold.
- **Incremental Learning:** Update the model continuously with new streaming data.
- **Champion-Challenger:** Keep current model (champion) live while testing a retrained model (challenger) on a small traffic slice before promoting it.

In [ ]:
# Example: Detecting data drift using Population Stability Index (PSI)
import numpy as np
import pandas as pd

def compute_psi(expected, actual, buckets=10):
    """Compute Population Stability Index between two distributions."""
    expected_perc = np.histogram(expected, bins=buckets)[0] / len(expected)
    actual_perc   = np.histogram(actual,   bins=buckets)[0] / len(actual)

    # Avoid division by zero / log(0)
    expected_perc = np.where(expected_perc == 0, 1e-6, expected_perc)
    actual_perc   = np.where(actual_perc   == 0, 1e-6, actual_perc)

    psi_value = np.sum((actual_perc - expected_perc) * np.log(actual_perc / expected_perc))
    return psi_value

# Simulate training vs production distributions
np.random.seed(42)
train_feature    = np.random.normal(loc=50, scale=10, size=1000)   # training distribution
prod_feature_ok  = np.random.normal(loc=51, scale=10, size=1000)   # slight shift
prod_feature_bad = np.random.normal(loc=70, scale=15, size=1000)   # significant drift

psi_ok  = compute_psi(train_feature, prod_feature_ok)
psi_bad = compute_psi(train_feature, prod_feature_bad)

print(f"PSI (minimal drift)  : {psi_ok:.4f}  → {'OK' if psi_ok < 0.1 else 'DRIFT DETECTED'}")
print(f"PSI (significant drift): {psi_bad:.4f}  → {'OK' if psi_bad < 0.1 else 'DRIFT DETECTED'}")
print()
print("PSI Thresholds: < 0.1 = No change | 0.1–0.2 = Moderate drift | > 0.2 = Significant drift")

## Question 10: High-Level System Architecture — ML Fraud Detection + GenAI Email Responses

### 1. Data Flow

Incoming transactions arrive in real-time via an **API gateway**. Raw transaction data (amount, merchant, location, device fingerprint, user history) is streamed through a **message queue (Apache Kafka)** to ensure decoupling and fault tolerance. This data is simultaneously:
- Written to a **Data Lake (e.g., AWS S3)** for historical storage and model retraining.
- Forwarded to the **ML Fraud Detection service** for real-time scoring.

### 2. Model Interaction

- **ML Fraud Detection Model:** A trained classifier (e.g., XGBoost or neural network) receives transaction features and returns a fraud probability score (0–1) in under 100ms. If the score exceeds a threshold (e.g., 0.85), the transaction is flagged as high-risk.
- **GenAI Email Response System:** When a transaction is flagged, the customer details are passed to an LLM (e.g., GPT-4 or a fine-tuned model) to generate a personalized email notifying the customer and requesting verification.
- The two models operate **independently** — the fraud model is a fast synchronous classifier; the GenAI module is an asynchronous pipeline that runs after the fraud decision.

### 3. Human-in-the-Loop

- Transactions with a fraud score in the **grey zone (0.60–0.85)** are routed to a human fraud analyst review queue rather than auto-blocked.
- The analyst sees: transaction details, model confidence score, SHAP explainability output, and the draft email.
- The analyst can **approve, modify, or discard** the generated email before it is sent.
- All analyst decisions are logged as labeled feedback to improve both models over time.

### 4. Monitoring Checkpoints

| Component | What to Monitor | Alert Threshold |
|---|---|---|
| **Fraud Model** | Detection rate, false positive rate, feature distribution drift | Accuracy drops > 5% from baseline |
| **GenAI Email Module** | Email open rates, customer dispute rate, human review sampling | 5% of sent emails reviewed weekly |
| **Latency** | End-to-end response time | Fraud scoring < 200ms; email < 5s |
| **Data Pipeline** | Kafka consumer lag, failed ingestion, schema validation errors | Any schema violation triggers alert |
| **Retraining Trigger** | PSI drift score on key features | Retrain if PSI > 0.2 or monthly |

In [ ]:
# High-level pseudocode for the Fraud Detection + GenAI Email architecture
import json
from dataclasses import dataclass
from typing import Optional

@dataclass
class Transaction:
    transaction_id: str
    customer_id: str
    amount: float
    merchant: str
    location: str
    customer_email: str

def fraud_detection_model(transaction: Transaction) -> float:
    """
    Returns a fraud probability score between 0 and 1.
    In production: calls a served ML model (e.g., XGBoost via FastAPI endpoint).
    """
    # Placeholder: replace with real model inference
    features = [transaction.amount, hash(transaction.merchant) % 100, hash(transaction.location) % 100]
    score = min(1.0, features[0] / 10000)   # simplified scoring for demo
    return round(score, 4)

def generate_fraud_alert_email(transaction: Transaction, score: float) -> str:
    """
    In production: calls an LLM API (e.g., OpenAI GPT-4 or Anthropic Claude)
    to generate a personalized customer email.
    """
    return (
        f"Dear Customer {transaction.customer_id},\n\n"
        f"We detected suspicious activity on your account.\n"
        f"Transaction ID: {transaction.transaction_id}\n"
        f"Amount: ${transaction.amount:.2f} at {transaction.merchant}\n"
        f"Risk Score: {score:.2f}\n\n"
        f"Please verify this transaction at your earliest convenience.\n\n"
        f"Regards,\nFraud Prevention Team"
    )

def process_transaction(transaction: Transaction):
    """Main pipeline: score → route → notify."""
    score = fraud_detection_model(transaction)
    print(f"Transaction {transaction.transaction_id} | Fraud Score: {score:.4f}")

    if score >= 0.85:
        email = generate_fraud_alert_email(transaction, score)
        print(f"AUTO-BLOCKED. Email sent to {transaction.customer_email}:\n{email}")
    elif 0.60 <= score < 0.85:
        print(f"GREY ZONE — Routed to human analyst queue for review.")
    else:
        print(f"Transaction approved.")

# Demo
txn = Transaction("TXN-001", "CUST-42", 8500.00, "Unknown Merchant", "Foreign Country", "user@example.com")
process_transaction(txn)